# RAG from scratch

**Track:** Enterprise Knowledge Assistant · **Stage:** Foundation

NovaTech wants an Enterprise Knowledge Assistant that can answer questions over finance reviews, HR policies, IT runbooks, project docs, and vendor contracts. In this first notebook you build the complete RAG loop manually before hiding it behind any framework. By the end, you should be able to explain every object in the loop: documents, chunks, retriever scores, selected context, answer, citation, abstention policy, and trace.

## What you will build

- A deterministic implementation that runs without API keys.
- A visible trace of evidence, decisions, and failure modes.
- A production design note explaining how this maps to real RAG libraries and systems.

## Concept map

```mermaid
flowchart LR
  D["Enterprise documents"] --> C["Parse + chunk"]
  C --> I["Index searchable chunks"]
  Q["User question"] --> R["Retrieve top evidence"]
  I --> R
  R --> G["Generate answer constrained by evidence"]
  G --> V["Validate citations or abstain"]
```

## Setup

Run this notebook from the repository root, or open it in GitHub and copy cells into a local Jupyter session. The helper code lives in `src/enterprise_rag` so the notebook remains readable while the implementation stays testable.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

def show(obj):
    print(json.dumps(obj, indent=2))

The failure-first question is: **what increased by 14% in Q2 2025?** A language model might know how to phrase a business answer, but it should not claim the NovaTech value unless retrieved evidence supports it. The implementation below compares an unsupported answer with a cited answer over retrieved chunks.

In [ ]:
from src.enterprise_rag.lab_experiments import build_enterprise_chunks, compare_rag_vs_no_rag
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
show(compare_rag_vs_no_rag("What increased by 14% in Q2 2025?", chunks))

### Architecture lesson

RAG is not “add a vector database.” It is a contract: the model may answer only from the evidence the system selected. The first production habit is to make the evidence inspectable. If you cannot show a reviewer which chunk supported the answer, you cannot debug whether the problem is retrieval, generation, stale data, or missing data.

## Deliberate failure case

Before moving on, make the system fail on purpose. Change one variable: chunk size, query wording, top-k, reranking terms, route choice, or evaluation labels. Write down whether the failure belongs to ingestion, retrieval, evidence selection, generation, authorization, or operations.

In [ ]:
# Try your own failure experiment here.
# Example: lower top_k to 1, ask an unsupported question, or remove an important query term.
from src.enterprise_rag.lab_experiments import build_enterprise_chunks
question = "What policy covers parental leave?"
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
print("Question:", question)
print("Now change the query, top_k, or chunking strategy and rerun a comparison helper.")

## Reflection questions

1. What did the simplest baseline get right?
2. What failure was invisible until you inspected the trace?
3. Which component would you improve first in production, and how would you prove it helped?
4. What should the system do when evidence is missing, unauthorized, stale, or contradictory?

## References and next reading

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*: https://arxiv.org/abs/2005.11401
- Stanford IR book: https://nlp.stanford.edu/IR-book/
- LangChain retrieval concepts: https://docs.langchain.com/oss/python/langchain/retrieval
- LlamaIndex RAG guide: https://docs.llamaindex.ai/en/stable/understanding/rag/
- Haystack pipeline docs: https://docs.haystack.deepset.ai/docs/pipelines
- Ragas metrics: https://docs.ragas.io/en/stable/concepts/metrics/